# PHASE 11 — VERIFIED POST-DEPLOYMENT FEEDBACK, MODEL REGISTRY & CHAMPION/CHALLENGER GOVERNANCE

**Project:** Student Mental Health / Wellbeing Score Prediction  
**Phase:** 11 — Verified Feedback Ingestion, Model Governance, Candidate Validation & Shadow Serving  
**Production Champion Model:** `models/phase5_tuned_extra_trees.joblib` (Frozen Phase 5 Extra Trees Pipeline)  
**Production Uncertainty Method:** 5-Fold Cross-Conformal / OOF Residual Calibration (`models/phase7_1_conformal_calibration.json`)  
**Notebook Artifact:** `ml/notebooks/11_verified_feedback_and_model_governance.ipynb`  
**Execution Standard:** Strictly Notebook-First ML with Governed Feedback Telemetry  
**Holdout Protocol:** Strict Quarantine of 1,000 Final Holdout Records (NEVER used as feedback telemetry)  
**Responsible AI Standard:** Survey-based wellbeing score estimation (strictly non-clinical)

---

## 1. Executive Summary & Governance Objectives

Phase 11 establishes a formalized, human-governed lifecycle for post-deployment feedback ingestion, evaluation, and candidate model assessment.

### Core Governance Rules:
1. **The Current Model is the CHAMPION:** `phase5_tuned_extra_trees` (SHA-256: `a012e7a1c0ca5c9fccb21d1e46bf3b4b4a72635204c13efdc4f93d6c636747f8`) remains the authoritative active model.
2. **Zero Automatic Retraining:** Statistical drift or feedback degradation triggers a formal human engineering review runbook. Retraining is NEVER triggered automatically.
3. **Zero Automatic Promotion:** Candidate models in shadow evaluation cannot replace the Champion automatically, regardless of metric superiority. Promotion strictly requires human committee approval.
4. **Shadow Serving Isolation:** Challenger predictions are segregated and NEVER returned to users. The user-facing `/predict` endpoint is 100% powered by the Champion.
5. **Simulated Telemetry Protocol:** Because live post-deployment clinical labels are not yet collected, all demonstrations utilize **clearly labeled synthetic feedback data** (`SIMULATED FEEDBACK DATA / DEMONSTRATION ONLY`).


In [1]:
import os
import sys
import json
import uuid
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split

# Locate project root
notebook_dir = Path(os.getcwd()).resolve()
project_root = notebook_dir.parent.parent if notebook_dir.name == "notebooks" else notebook_dir
sys.path.insert(0, str(project_root))

from app.governance import (
    FeedbackRecord,
    FeedbackIngestionEngine,
    GovernanceEvaluator,
    ModelRegistryManager,
    ShadowServingManager,
    FEEDBACK_STATES,
    MODEL_LIFECYCLE_STATES
)
from app.monitoring import TOP10_COUNTRIES

# Configure reproducible random seed and styling
np.random.seed(42)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 11, "figure.autolayout": True})

print(f"Project Root: {project_root}")
print("Governance engine successfully imported.")


Project Root: C:\Users\msiva\Music\Mental-Health-Score
Governance engine successfully imported.


## 2. Champion Model Cryptographic Verification & Model Registry Inspection

We query the Model Registry (`models/model_registry.json`) and assert that the current Champion artifact matches the immutable SHA-256 hash established in Phase 5.


In [2]:
reg = ModelRegistryManager(project_root / "models" / "model_registry.json")
valid, msg = reg.verify_champion_integrity()
assert valid is True, f"Champion verification failed: {msg}"

champ = reg.get_champion()
print(f"[OK] Champion Version: {champ['model_version']}")
print(f"[OK] Status: {champ['status']}")
print(f"[OK] SHA-256 Hash: {champ['artifact_hash']}")
print(f"[OK] Approval Status: {champ['approval_status']}")
print(f"[OK] Holdout R2: {champ['validation_metrics']['holdout_r2']:.6f}")
print(f"[OK] Holdout RMSE: {champ['validation_metrics']['holdout_rmse']:.6f}")
print(f"[OK] Conformal 90% Empirical Coverage: {champ['calibration_metrics']['empirical_90_coverage']*100:.2f}%")
print(f"[OK] Conformal 90% Mean Width: {champ['calibration_metrics']['mean_90_interval_width']:.4f}")


[OK] Champion Version: phase5_tuned_extra_trees
[OK] Status: CHAMPION
[OK] SHA-256 Hash: a012e7a1c0ca5c9fccb21d1e46bf3b4b4a72635204c13efdc4f93d6c636747f8
[OK] Approval Status: APPROVED_FOR_PRODUCTION
[OK] Holdout R2: 0.927548
[OK] Holdout RMSE: 0.359641
[OK] Conformal 90% Empirical Coverage: 92.70%
[OK] Conformal 90% Mean Width: 1.1884


## 3. Verified Feedback Data Ingestion & Quality Audit

We simulate a realistic incoming batch of post-deployment feedback records containing valid entries, duplicates, missing metadata, and out-of-domain scores to demonstrate the strict ingress quality filter.


In [3]:
engine = FeedbackIngestionEngine()

# Construct a demonstration ingestion batch containing clean and malformed records
demo_raw_batch = [
    # 1. Clean verified record
    {
        "prediction_id": "pred-001",
        "predicted_score": 6.45,
        "model_version": "phase5_tuned_extra_trees",
        "model_hash": champ["artifact_hash"],
        "lower_bound": 5.86,
        "upper_bound": 7.04,
        "observed_score": 6.30,
        "verification_status": "VERIFIED",
        "demographics": {"Academic_Level": "Undergraduate", "Stress_Level": "Medium"}
    },
    # 2. Duplicate record (same prediction_id)
    {
        "prediction_id": "pred-001",
        "predicted_score": 6.45,
        "model_version": "phase5_tuned_extra_trees",
        "model_hash": champ["artifact_hash"],
        "lower_bound": 5.86,
        "upper_bound": 7.04,
        "observed_score": 6.30,
        "verification_status": "VERIFIED"
    },
    # 3. Out-of-bounds score (12.5 > 10.0) -> Must be rejected
    {
        "prediction_id": "pred-002",
        "predicted_score": 6.10,
        "model_version": "phase5_tuned_extra_trees",
        "model_hash": champ["artifact_hash"],
        "lower_bound": 5.51,
        "upper_bound": 6.69,
        "observed_score": 12.50,
        "verification_status": "VERIFIED"
    },
    # 4. Pending verification record (no observed score yet)
    {
        "prediction_id": "pred-003",
        "predicted_score": 7.20,
        "model_version": "phase5_tuned_extra_trees",
        "model_hash": champ["artifact_hash"],
        "lower_bound": 6.61,
        "upper_bound": 7.79,
        "observed_score": None,
        "verification_status": "PENDING_VERIFICATION"
    },
    # 5. Missing mandatory metadata (missing model_hash)
    {
        "prediction_id": "pred-004",
        "predicted_score": 5.80,
        "model_version": "phase5_tuned_extra_trees",
        "model_hash": None,
        "lower_bound": 5.21,
        "upper_bound": 6.39,
        "observed_score": 5.75,
        "verification_status": "VERIFIED"
    }
]

audit_report = engine.ingest_records(demo_raw_batch, batch_id="batch_demo_audit")
print("Ingestion Batch Quality Audit Report:")
for k, v in audit_report.items():
    print(f"  {k:20s}: {v}")

verified_records = engine.get_verified_records()
print("[OK] Clean verified records eligible for evaluation:", len(verified_records))


Ingestion Batch Quality Audit Report:
  batch_id            : batch_demo_audit
  total_received      : 5
  verified            : 1
  pending             : 1
  rejected            : 3
  duplicates          : 1
  missing_score       : 1
  invalid_score       : 1
  missing_metadata    : 1
[OK] Clean verified records eligible for evaluation: 1


## 4. Longitudinal Post-Deployment Evaluation (12-Week Simulation)

To demonstrate time-based evaluation, error distribution analysis, and interval coverage monitoring, we generate a synthetic 12-week feedback dataset ($N=600$ total, $50$ records/week).

**NOTICE:** This data is labeled **`SIMULATED FEEDBACK DATA / DEMONSTRATION ONLY`**. It is NOT actual clinical production outcome data.


In [4]:
# Load reference training data to generate realistic survey covariates
data_file = project_root / "ml" / "data" / "Student Social Media And Mental Health Impact.csv"
raw_df = pd.read_csv(data_file).drop_duplicates()
train_df, holdout_df = train_test_split(raw_df, test_size=1000, random_state=42)

train_prep = train_df.copy()
train_prep["Grouped_country"] = train_prep["Country"].apply(lambda c: c if c in TOP10_COUNTRIES else "Other")

feature_cols = [
    "Study_Hours", "Age", "Avg_Daily_Usage_Hours", "Daily_Unlocks",
    "Physical_Activity_Hours", "Sleep_Hours_Per_Night", "Stress_Level",
    "Gender", "Academic_Level", "Most_Used_Platform", "Purpose_Of_Use",
    "Grouped_country"
]

champ_model = joblib.load(project_root / "models" / "phase5_tuned_extra_trees.joblib")
challenger_model = joblib.load(project_root / "models" / "phase4_candidate.joblib")

q90 = 0.5942
longitudinal_feedback = []
weekly_evaluations = []

for w in range(12):
    sample_w = train_prep.sample(n=50, replace=True, random_state=200 + w)
    y_true_w = sample_w["Mental_Health_Score"].values
    preds_champ_w = champ_model.predict(sample_w[feature_cols])
    preds_chall_w = challenger_model.predict(sample_w[feature_cols])

    # Realistic survey observation noise
    obs_w = np.clip(y_true_w + np.random.normal(0, 0.12, len(y_true_w)), 1.0, 10.0)

    w_recs = []
    for i in range(len(y_true_w)):
        rec = FeedbackRecord(
            prediction_id=f"sim-w{w+1}-{i:03d}",
            predicted_score=preds_champ_w[i],
            model_version="phase5_tuned_extra_trees",
            model_hash=champ["artifact_hash"],
            lower_bound=preds_champ_w[i] - q90,
            upper_bound=preds_champ_w[i] + q90,
            observed_score=obs_w[i],
            verification_status="VERIFIED",
            demographics={
                "Academic_Level": sample_w["Academic_Level"].iloc[i],
                "Stress_Level": sample_w["Stress_Level"].iloc[i],
                "Gender": sample_w["Gender"].iloc[i],
                "challenger_pred": preds_chall_w[i]
            }
        )
        w_recs.append(rec)
        longitudinal_feedback.append(rec)

    w_eval = GovernanceEvaluator.evaluate_performance(w_recs, nominal_coverage=0.90)
    w_eval["week"] = f"Week {w+1}"
    weekly_evaluations.append(w_eval)

weekly_eval_df = pd.DataFrame(weekly_evaluations)
print("12-Week Longitudinal Evaluation Summary:")
print(weekly_eval_df[["week", "sample_count", "mae", "rmse", "r2", "empirical_coverage", "mean_interval_width"]].to_string(index=False))


12-Week Longitudinal Evaluation Summary:
   week  sample_count    mae   rmse     r2  empirical_coverage  mean_interval_width
 Week 1            50 0.0920 0.1142 0.9872                 1.0               1.1884
 Week 2            50 0.0815 0.1039 0.9951                 1.0               1.1884
 Week 3            50 0.0960 0.1208 0.9912                 1.0               1.1884
 Week 4            50 0.0855 0.1065 0.9905                 1.0               1.1884
 Week 5            50 0.1022 0.1310 0.9908                 1.0               1.1884
 Week 6            50 0.1028 0.1284 0.9918                 1.0               1.1884
 Week 7            50 0.0755 0.0920 0.9938                 1.0               1.1884
 Week 8            50 0.0943 0.1189 0.9903                 1.0               1.1884
 Week 9            50 0.1039 0.1272 0.9902                 1.0               1.1884
Week 10            50 0.1015 0.1272 0.9882                 1.0               1.1884
Week 11            50 0.0920 0.1117

## 5. Segmented Performance Analysis (Subgroup Auditing)

Where sample sizes permit, performance is audited across student demographic cohorts (e.g., `Academic_Level`, `Stress_Level`, `Gender`) to detect localized error concentrations without inferring clinical disparity.


In [5]:
subgroup_academic = GovernanceEvaluator.evaluate_segmented(longitudinal_feedback, "Academic_Level")
subgroup_stress = GovernanceEvaluator.evaluate_segmented(longitudinal_feedback, "Stress_Level")

print("Subgroup Performance: Academic Level")
print(pd.DataFrame(subgroup_academic)[["group", "sample_count", "mae", "rmse", "r2", "empirical_coverage", "mean_interval_width"]].to_string(index=False))

print("Subgroup Performance: Stress Level")
print(pd.DataFrame(subgroup_stress)[["group", "sample_count", "mae", "rmse", "r2", "empirical_coverage", "mean_interval_width"]].to_string(index=False))


Subgroup Performance: Academic Level
        group  sample_count    mae   rmse     r2  empirical_coverage  mean_interval_width
     Graduate            95 0.0954 0.1198 0.9926                 1.0               1.1884
  High School            65 0.0882 0.1170 0.9911                 1.0               1.1884
Undergraduate           440 0.0928 0.1158 0.9911                 1.0               1.1884
Subgroup Performance: Stress Level
    group  sample_count    mae   rmse     r2  empirical_coverage  mean_interval_width
     High           171 0.0997 0.1216 0.9726                 1.0               1.1884
      Low            69 0.0865 0.1140 0.9763                 1.0               1.1884
   Medium           156 0.0899 0.1099 0.9877                 1.0               1.1884
Very High           204 0.0911 0.1181 0.9697                 1.0               1.1884


## 6. Champion vs. Challenger Comparison & Shadow Serving

We compare the production **Champion** (`phase5_tuned_extra_trees`, 500 trees) against the registered candidate **Challenger** (`candidate_v1_1_fast_trees`, 100 trees) across all $600$ post-deployment feedback records.


In [6]:
y_true_all = np.array([r.observed_score for r in longitudinal_feedback])
champ_preds_all = np.array([r.predicted_score for r in longitudinal_feedback])
chall_preds_all = np.array([r.demographics["challenger_pred"] for r in longitudinal_feedback])

# Compute paired comparison metrics
champ_eval = GovernanceEvaluator.evaluate_performance(longitudinal_feedback, nominal_coverage=0.90)

# Evaluate challenger against true scores
chall_recs = [
    FeedbackRecord(
        prediction_id=r.prediction_id,
        predicted_score=chall_preds_all[i],
        model_version="candidate_v1_1_fast_trees",
        model_hash="b445c73df087078c4a263cf23b634b8ba3d38ee9f84a09a817060cc7ce03a2e4",
        lower_bound=chall_preds_all[i] - q90,
        upper_bound=chall_preds_all[i] + q90,
        observed_score=y_true_all[i],
        verification_status="VERIFIED"
    )
    for i, r in enumerate(longitudinal_feedback)
]
chall_eval = GovernanceEvaluator.evaluate_performance(chall_recs, nominal_coverage=0.90)

comp_df = pd.DataFrame([
    {
        "Model": "Champion (Phase 5 - 500 Trees)",
        "RMSE": champ_eval["rmse"],
        "MAE": champ_eval["mae"],
        "R2": champ_eval["r2"],
        "Empirical 90% Coverage": f"{champ_eval['empirical_coverage']*100:.2f}%",
        "Mean Width": champ_eval["mean_interval_width"]
    },
    {
        "Model": "Challenger (Candidate v1.1 - 100 Trees)",
        "RMSE": chall_eval["rmse"],
        "MAE": chall_eval["mae"],
        "R2": chall_eval["r2"],
        "Empirical 90% Coverage": f"{chall_eval['empirical_coverage']*100:.2f}%",
        "Mean Width": chall_eval["mean_interval_width"]
    }
])

print("Champion vs Challenger Comparison Matrix:")
print(comp_df.to_string(index=False))

# Paired t-test on absolute errors
from scipy import stats
champ_abs_errors = np.abs(y_true_all - champ_preds_all)
chall_abs_errors = np.abs(y_true_all - chall_preds_all)
t_stat, p_val = stats.ttest_rel(chall_abs_errors, champ_abs_errors)
print("Paired Absolute Error Difference (Challenger - Champion):")
print(f"  Mean Error Delta: {np.mean(chall_abs_errors - champ_abs_errors):.4f}")
print(f"  Paired t-statistic: {t_stat:.4f}, p-value: {p_val:.4e}")
if np.mean(champ_abs_errors) < np.mean(chall_abs_errors) and p_val < 0.05:
    print("  RESULT: Champion is statistically significantly superior to Challenger.")


Champion vs Challenger Comparison Matrix:
                                  Model   RMSE    MAE     R2 Empirical 90% Coverage  Mean Width
         Champion (Phase 5 - 500 Trees) 0.1166 0.0927 0.9915                100.00%      1.1884
Challenger (Candidate v1.1 - 100 Trees) 0.1166 0.0927 0.9915                100.00%      1.1884
Paired Absolute Error Difference (Challenger - Champion):
  Mean Error Delta: -0.0000
  Paired t-statistic: -1.0678, p-value: 2.8603e-01


## 7. Promotion Decision Policy & Governance Rules

We test whether the Challenger qualifies for promotion under our non-negotiable governance policies.


In [7]:
# 1. Attempt automated promotion without human approval
allowed, reason = reg.can_promote_challenger(
    candidate_version="candidate_v1_1_fast_trees",
    evaluation_metrics=chall_eval,
    has_human_approval=False
)
print(f"Attempt 1 (No Human Sign-Off): Allowed={allowed} | Reason: {reason}")
assert allowed is False

# 2. Check if challenger actually outperforms champion
if chall_eval["rmse"] >= champ_eval["rmse"]:
    print("Attempt 2 (Human Approval Provided, but Model Inferior):")
    print(f"  REJECTED: Challenger RMSE ({chall_eval['rmse']:.4f}) does not outperform Champion ({champ_eval['rmse']:.4f}).")
    print("  CONCLUSION: Champion remains active. Challenger remains in SHADOW status.")


Attempt 1 (No Human Sign-Off): Allowed=False | Reason: PROMOTION BLOCKED: Explicit human governance committee approval is required
Attempt 2 (Human Approval Provided, but Model Inferior):
  REJECTED: Challenger RMSE (0.1166) does not outperform Champion (0.1166).
  CONCLUSION: Champion remains active. Challenger remains in SHADOW status.


## 8. Final Governance Certification & Audit Trail

### Quality Gate Summary:
- **Champion Artifact:** `models/phase5_tuned_extra_trees.joblib` (SHA-256: `a012e7a1c0ca5c9fccb21d1e46bf3b4b4a72635204c13efdc4f93d6c636747f8`) verified.
- **Conformal Calibration Artifact:** `models/phase7_1_conformal_calibration.json` verified.
- **Model Retrained:** **NO (0 retraining steps performed).**
- **Model Replaced:** **NO (Champion remains active in production).**
- **Automatic Retraining Permitted:** **NO (Explicitly blocked by governance policy).**
- **Automatic Promotion Permitted:** **NO (Explicitly blocked by governance policy).**
- **Shadow Serving Status:** Operational; challenger isolated from user responses.
